# 🇦🇺 Rain in Australia - Dataset Processing Pipeline

## Dataset Information

**Dataset Name**: Rain in Australia  
**Source**: Kaggle (https://www.kaggle.com/datasets/jsphyg/weather-dataset-raining-in-australia)  
**Original Data Source**: Bureau of Meteorology (BOM)  
**Copyright**: Commonwealth of Australia 2010

### Context
Ever wondered if you should carry an umbrella tomorrow? With this dataset, you can predict next-day rain by training classification models on the target variable `RainTomorrow`.

### Content
- **Timespan**: ~10 years of daily weather observations
- **Locations**: Multiple weather stations across Australia
- **Target Variable**: `RainTomorrow` (Yes/No) - Will it rain the next day?
- **Definition**: Column marked 'Yes' if rain for that day was ≥1mm

### Data Source
- Daily observations from Bureau of Meteorology weather stations
- Latest observations available at: http://www.bom.gov.au/climate/data
- Example: [Canberra Weather Data](http://www.bom.gov.au/climate/data)

### Original Citation
Bureau of Meteorology Climate Data Online - https://www.bom.gov.au/climate/data

---

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

print("✅ Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Load Raw Data

Load the dataset from the raw folder. You can download from Kaggle using:
```bash
kaggle datasets download -d jsphyg/weather-dataset-raining-in-australia
```

Or download directly from: https://www.kaggle.com/datasets/jsphyg/weather-dataset-raining-in-australia

In [ ]:
# Define dataset paths
dataset_path = Path('.')
raw_path = dataset_path / 'raw'
processed_path = dataset_path / 'processed'
features_path = dataset_path / 'features'

# Create directories if they don't exist
raw_path.mkdir(exist_ok=True)
processed_path.mkdir(exist_ok=True)
features_path.mkdir(exist_ok=True)

# Load the raw data
# Note: Download weatherAUS.csv from Kaggle and place in raw/ folder
try:
    # Try to load the CSV file
    df_raw = pd.read_csv(raw_path / 'weatherAUS.csv')
    print(f"✅ Loaded raw data successfully!")
except FileNotFoundError:
    # Create sample data for demonstration
    print("⚠️ weatherAUS.csv not found. Creating sample data for demonstration...")
    dates = pd.date_range(start='2015-01-01', periods=1000, freq='D')
    df_raw = pd.DataFrame({
        'Date': dates,
        'Location': np.random.choice(['Sydney', 'Melbourne', 'Brisbane', 'Perth', 'Adelaide'], 1000),
        'MinTemp': np.random.uniform(5, 20, 1000),
        'MaxTemp': np.random.uniform(20, 40, 1000),
        'Rainfall': np.random.exponential(scale=5, size=1000),
        'Evaporation': np.random.uniform(0, 15, 1000),
        'Sunshine': np.random.uniform(0, 14, 1000),
        'WindGustDir': np.random.choice(['N', 'S', 'E', 'W', 'NE', 'NW', 'SE', 'SW'], 1000),
        'WindGustSpeed': np.random.uniform(10, 70, 1000),
        'WindDir9am': np.random.choice(['N', 'S', 'E', 'W', 'NE', 'NW', 'SE', 'SW'], 1000),
        'WindDir3pm': np.random.choice(['N', 'S', 'E', 'W', 'NE', 'NW', 'SE', 'SW'], 1000),
        'WindSpeed9am': np.random.uniform(0, 40, 1000),
        'WindSpeed3pm': np.random.uniform(0, 40, 1000),
        'Humidity9am': np.random.randint(20, 100, 1000),
        'Humidity3pm': np.random.randint(20, 100, 1000),
        'Pressure9am': np.random.uniform(1000, 1030, 1000),
        'Pressure3pm': np.random.uniform(1000, 1030, 1000),
        'Cloud9am': np.random.randint(0, 9, 1000),
        'Cloud3pm': np.random.randint(0, 9, 1000),
        'Temp9am': np.random.uniform(10, 30, 1000),
        'Temp3pm': np.random.uniform(10, 35, 1000),
        'RainToday': np.random.choice(['Yes', 'No'], 1000),
        'RainTomorrow': np.random.choice(['Yes', 'No'], 1000, p=[0.3, 0.7])
    })

# Display dataset info
print(f"\n📊 Dataset Shape: {df_raw.shape}")
print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")
print(f"\n📋 First few rows:")
print(df_raw.head())
print(f"\n📋 Data types:")
print(df_raw.dtypes)
print(f"\n📋 Missing values:")
print(df_raw.isnull().sum())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Statistical summary
print("📊 STATISTICAL SUMMARY")
print("=" * 80)
print(df_raw.describe())

# Null values analysis
print("\n\n🔍 MISSING VALUES ANALYSIS")
print("=" * 80)
missing_stats = pd.DataFrame({
    'Column': df_raw.columns,
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Percentage': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
print(missing_stats[missing_stats['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False))

# Target variable distribution
print("\n\n🎯 TARGET VARIABLE DISTRIBUTION")
print("=" * 80)
if 'RainTomorrow' in df_raw.columns:
    print(df_raw['RainTomorrow'].value_counts())
    print(f"\nRain Probability: {(df_raw['RainTomorrow'] == 'Yes').sum() / len(df_raw) * 100:.2f}%")

# Unique locations
print("\n\n📍 UNIQUE LOCATIONS")
print("=" * 80)
if 'Location' in df_raw.columns:
    print(f"Number of unique locations: {df_raw['Location'].nunique()}")
    print(df_raw['Location'].value_counts().head(10))

## 4. Data Cleaning & Preprocessing

In [ ]:
# Create a copy for processing
df_clean = df_raw.copy()

print("🧹 STARTING DATA CLEANING PROCESS")
print("=" * 80)

# Step 1: Remove duplicates
duplicates_before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()
duplicates_removed = duplicates_before - df_clean.shape[0]
print(f"✓ Step 1: Removed {duplicates_removed} duplicate rows")

# Step 2: Standardize column names
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(' ', '_')
print(f"✓ Step 2: Standardized column names")

# Step 3: Handle missing values
print(f"\n✓ Step 3: Handling missing values")

# Numeric columns - fill with median
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"  - Filled '{col}' with median: {median_val:.2f}")

# Categorical columns - fill with mode
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0] if len(df_clean[col].mode()) > 0 else 'Unknown'
        df_clean[col].fillna(mode_val, inplace=True)
        print(f"  - Filled '{col}' with mode: {mode_val}")

# Step 4: Remove rows where target variable is missing
if 'raintomorrow' in df_clean.columns:
    rows_before = df_clean.shape[0]
    df_clean = df_clean[df_clean['raintomorrow'].notna()]
    rows_removed = rows_before - df_clean.shape[0]
    print(f"\n✓ Step 4: Removed {rows_removed} rows with missing target variable")

# Step 5: Handle outliers using IQR method
print(f"\n✓ Step 5: Removing outliers (IQR method)")
for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)].shape[0]
    if outliers > 0:
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
        print(f"  - Removed {outliers} outliers from '{col}'")

# Step 6: Encode categorical variables
print(f"\n✓ Step 6: Encoding categorical variables")
label_encoders = {}
categorical_features = df_clean.select_dtypes(include=['object']).columns

for col in categorical_features:
    if col not in ['date']:  # Don't encode date columns
        le = LabelEncoder()
        df_clean[col + '_encoded'] = le.fit_transform(df_clean[col].astype(str))
        label_encoders[col] = le
        print(f"  - Encoded '{col}' with {len(le.classes_)} unique values")

print(f"\n✅ DATA CLEANING COMPLETE!")
print(f"Final shape: {df_clean.shape}")
print(f"Final missing values: {df_clean.isnull().sum().sum()}")

## 5. Feature Engineering

In [ ]:
df_features = df_clean.copy()

print("⚙️ FEATURE ENGINEERING")
print("=" * 80)

# 1. Temperature features
if 'mintemp' in df_features.columns and 'maxtemp' in df_features.columns:
    df_features['temp_range'] = df_features['maxtemp'] - df_features['mintemp']
    df_features['temp_avg'] = (df_features['maxtemp'] + df_features['mintemp']) / 2
    print("✓ Created temperature features (temp_range, temp_avg)")

# 2. Wind features
if 'windspeed9am' in df_features.columns and 'windspeed3pm' in df_features.columns:
    df_features['wind_speed_diff'] = df_features['windspeed3pm'] - df_features['windspeed9am']
    df_features['wind_speed_avg'] = (df_features['windspeed9am'] + df_features['windspeed3pm']) / 2
    print("✓ Created wind features (wind_speed_diff, wind_speed_avg)")

# 3. Humidity features
if 'humidity9am' in df_features.columns and 'humidity3pm' in df_features.columns:
    df_features['humidity_diff'] = df_features['humidity3pm'] - df_features['humidity9am']
    df_features['humidity_avg'] = (df_features['humidity9am'] + df_features['humidity3pm']) / 2
    print("✓ Created humidity features (humidity_diff, humidity_avg)")

# 4. Pressure features
if 'pressure9am' in df_features.columns and 'pressure3pm' in df_features.columns:
    df_features['pressure_diff'] = df_features['pressure3pm'] - df_features['pressure9am']
    print("✓ Created pressure features (pressure_diff)")

# 5. Rainfall features
if 'rainfall' in df_features.columns:
    df_features['has_rainfall'] = (df_features['rainfall'] > 0).astype(int)
    df_features['is_heavy_rain'] = (df_features['rainfall'] > 10).astype(int)
    print("✓ Created rainfall features (has_rainfall, is_heavy_rain)")

# 6. Cloud features
if 'cloud9am' in df_features.columns and 'cloud3pm' in df_features.columns:
    df_features['cloud_diff'] = df_features['cloud3pm'] - df_features['cloud9am']
    df_features['cloud_avg'] = (df_features['cloud9am'] + df_features['cloud3pm']) / 2
    print("✓ Created cloud features (cloud_diff, cloud_avg)")

print(f"\n✅ Feature engineering complete!")
print(f"Total features created: {df_features.shape[1]}")
print(f"\nNew features:")
for col in df_features.columns:
    if col not in df_clean.columns:
        print(f"  - {col}")

## 6. Prepare Data for Machine Learning

In [ ]:
print("🔧 PREPARING DATA FOR MACHINE LEARNING")
print("=" * 80)

# Prepare features and target
df_ml = df_features.copy()

# Target variable
target_col = 'raintomorrow'
if target_col in df_ml.columns:
    y = df_ml[target_col + '_encoded']  # Use encoded target
    print(f"✓ Target variable: {target_col}")
    print(f"  - Class distribution: {dict(df_ml[target_col].value_counts())}")
else:
    print("⚠️ Target variable not found!")
    y = None

# Select features (exclude target and original categorical columns)
exclude_cols = [target_col, 'date', 'location'] + [col for col in df_ml.columns if col.endswith('_encoded') and col != target_col + '_encoded']
X = df_ml.drop(columns=[col for col in exclude_cols if col in df_ml.columns])

# Select only numeric features for ML
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
X = X[numeric_features]

print(f"\n✓ Selected {len(numeric_features)} numeric features")
print(f"  Features: {numeric_features[:10]}{'...' if len(numeric_features) > 10 else ''}")

# Train/Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if y is not None else None
)

print(f"\n✓ Train/Test Split (80-20)")
print(f"  - Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"  - Testing set: {X_test.shape[0]} samples")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Standardized features using StandardScaler")

# Create DataFrames with feature names
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=numeric_features)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=numeric_features)

print(f"\n✅ ML DATA PREPARATION COMPLETE!")
print(f"  Training set shape: {X_train_scaled_df.shape}")
print(f"  Testing set shape: {X_test_scaled_df.shape}")

## 7. Save Processed Datasets

In [ ]:
print("💾 SAVING PROCESSED DATASETS")
print("=" * 80)

# Save cleaned data
df_clean.to_csv(processed_path / 'rain_australia_clean.csv', index=False)
print(f"✓ Saved: {processed_path / 'rain_australia_clean.csv'}")

# Save engineered features
df_features.to_csv(processed_path / 'rain_australia_features.csv', index=False)
print(f"✓ Saved: {processed_path / 'rain_australia_features.csv'}")

# Save ML-ready datasets
X_train_scaled_df.to_csv(features_path / 'X_train_scaled.csv', index=False)
X_test_scaled_df.to_csv(features_path / 'X_test_scaled.csv', index=False)
y_train.to_csv(features_path / 'y_train.csv', index=False, header=['target'])
y_test.to_csv(features_path / 'y_test.csv', index=False, header=['target'])
print(f"✓ Saved: {features_path / 'X_train_scaled.csv'}")
print(f"✓ Saved: {features_path / 'X_test_scaled.csv'}")
print(f"✓ Saved: {features_path / 'y_train.csv'}")
print(f"✓ Saved: {features_path / 'y_test.csv'}")

# Save feature scaler for production use
import pickle
with open(features_path / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f"✓ Saved: {features_path / 'scaler.pkl'}")

# Create metadata file
metadata = {
    "dataset_name": "Rain in Australia",
    "source": "Kaggle - https://www.kaggle.com/datasets/jsphyg/weather-dataset-raining-in-australia",
    "original_source": "Bureau of Meteorology (BOM)",
    "copyright": "Commonwealth of Australia 2010",
    "description": "10 years of daily weather observations from multiple locations across Australia to predict next-day rain",
    "target_variable": "RainTomorrow",
    "target_values": ["Yes", "No"],
    "created_date": str(datetime.now()),
    "data_cleaning": {
        "duplicates_removed": int(duplicates_removed),
        "missing_values_handled": "Median for numeric, Mode for categorical",
        "outliers_removed_method": "IQR",
        "categorical_encoding": "LabelEncoder"
    },
    "features_engineered": [
        "temp_range", "temp_avg",
        "wind_speed_diff", "wind_speed_avg",
        "humidity_diff", "humidity_avg",
        "pressure_diff",
        "has_rainfall", "is_heavy_rain",
        "cloud_diff", "cloud_avg"
    ],
    "total_samples": len(df_clean),
    "total_features": len(numeric_features),
    "feature_list": numeric_features,
    "train_test_split": {
        "train_size": len(X_train_scaled_df),
        "test_size": len(X_test_scaled_df),
        "split_ratio": "80-20",
        "stratified": True
    },
    "preprocessing": {
        "method": "StandardScaler",
        "scaler_file": "scaler.pkl"
    },
    "files": {
        "raw": "raw/weatherAUS.csv",
        "cleaned": "processed/rain_australia_clean.csv",
        "features": "processed/rain_australia_features.csv",
        "X_train": "features/X_train_scaled.csv",
        "X_test": "features/X_test_scaled.csv",
        "y_train": "features/y_train.csv",
        "y_test": "features/y_test.csv",
        "scaler": "features/scaler.pkl",
        "metadata": "metadata.json"
    }
}

# Save metadata
with open(dataset_path / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Saved: {dataset_path / 'metadata.json'}")

print(f"\n✅ ALL DATASETS SAVED SUCCESSFULLY!")
print(f"\n📁 Dataset Structure:")
print(f"   raw/")
print(f"   ├── weatherAUS.csv (original data)")
print(f"   processed/")
print(f"   ├── rain_australia_clean.csv (cleaned)")
print(f"   ├── rain_australia_features.csv (with engineered features)")
print(f"   features/")
print(f"   ├── X_train_scaled.csv")
print(f"   ├── X_test_scaled.csv")
print(f"   ├── y_train.csv")
print(f"   ├── y_test.csv")
print(f"   └── scaler.pkl")
print(f"   metadata.json (dataset documentation)")

## 8. Quick Start - Using the ML-Ready Data

### Load the preprocessed data:
```python
import pandas as pd
from sklearn.preprocessing import StandardScaler
import pickle

# Load ML-ready data
X_train = pd.read_csv('features/X_train_scaled.csv')
X_test = pd.read_csv('features/X_test_scaled.csv')
y_train = pd.read_csv('features/y_train.csv').squeeze()
y_test = pd.read_csv('features/y_test.csv').squeeze()

# Load scaler if needed
with open('features/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
```

### Train a model:
```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
```

### Metadata:
Load `metadata.json` to understand the dataset structure and preprocessing steps.

---

## Summary

✅ **Data Cleaned**: Duplicates removed, missing values handled, outliers removed  
✅ **Features Engineered**: 10+ new features created from raw data  
✅ **ML-Ready**: Scaled, split (80-20), ready for model training  
✅ **Documented**: Metadata file with all processing details  
✅ **Reproducible**: Scaler saved for production predictions